<a id="top"></a>

# Lab L1005: tool failures are messages, not exceptions

```yaml
title: "Lab L1005: tool failures are messages, not exceptions"
keywords: tool failure, dispatch, exception to ToolMessage, status error, structured error, unknown tool, recovery, offline, lab
estimated duration: 35
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md). Solutions: `L1005_lab_solutions.ipynb`. **Pure Python — no API key needed.** You build the loop's failure-handling layer against a *scripted stub model*, so a tool that raises is exercised deterministically.
>
> **Builds on [L08](../L08/objectives.md):** L08 taught the *tool author* what to **return** on failure (errors as data). This lab is the *loop* layer — what to do when the tool **can't even return** (it raises, or the name is unknown).
>
> **After this lab you can:** convert a tool exception into a `ToolMessage` with `status="error"`, pass a structured (already-handled) error through unchanged, and watch the model recover — all without crashing the loop.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write dispatch: turn a raise into a ToolMessage](#2-problem-1--write-dispatch-turn-a-raise-into-a-toolmessage)
- [3. Problem 2 — The three failure modes, one by one](#3-problem-2--the-three-failure-modes-one-by-one)
- [4. Problem 3 — Watch the model recover (no crash)](#4-problem-3--watch-the-model-recover-no-crash)
- [5. Problem 4 — Why not dump the traceback? (written)](#5-problem-4--why-not-dump-the-traceback-written)
- [6. Problem 5 — Should the loop auto-retry? (written)](#6-problem-5--should-the-loop-auto-retry-written)
- [7. Self-check](#7-self-check)

## 1. Setup

Given: the same tools (`calculator` raises `ValueError`, `lookup` raises `KeyError` on a missing city), the scripted `FakeModel`, the `RunResult` dataclass, and — already complete — the `run_loop` you wrote in [L1004](L1004_lab_empty.ipynb). You write only `dispatch`, the loop's failure-handling layer: it turns each tool call into a `ToolMessage`.

In [ ]:
from collections.abc import Callable
from dataclasses import dataclass

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.messages.tool import ToolCall

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# Dispatch table: tool NAME -> the function that runs it.
TOOLS: dict[str, Callable[..., str]] = {"calculator": calculator, "lookup": lookup}

In [ ]:
@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"

In [ ]:
def run_loop(
    model: object, tools: dict[str, Callable[..., str]], user_msg: str, max_steps: int
) -> RunResult:
    """The loop from L1004 (given here). It calls dispatch() -- which YOU write below."""
    bound = model.bind_tools(list(tools.values()))
    messages: list[BaseMessage] = [HumanMessage(content=user_msg)]
    for step in range(1, max_steps + 1):
        reply = bound.invoke(messages)
        if not reply.tool_calls:
            return RunResult(reply.text, step, "natural")
        messages.append(reply)
        for call in reply.tool_calls:
            messages.append(dispatch(tools, call))
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 2. Problem 1 — Write dispatch: turn a raise into a ToolMessage

`dispatch(tools, call)` runs one requested tool and **always returns a `ToolMessage`** — it never lets an exception escape. A `call` here is a LangChain `ToolCall` dict: `{"name", "args", "id"}`. Three cases:

1. **Unknown tool name** (`tools.get(call["name"])` is `None`): return a `ToolMessage` with `status="error"` and content like `no such tool 'X'`.
2. **Tool runs cleanly:** `ToolMessage(content=output, tool_call_id=call["id"], status="success")`.
3. **Tool raises:** catch the exception and return a `ToolMessage` with `status="error"` and `content=repr(exc)` — a **short** message (class + one line), **never** a traceback.

Use `fn(**call["args"])` to call the tool with the model's arguments.

In [ ]:
def dispatch(tools: dict[str, Callable[..., str]], call: ToolCall) -> ToolMessage:
    # TODO case 1: unknown name -> ToolMessage(status='error', 'no such tool ...').
    # TODO case 2: success     -> ToolMessage(content=output, status='success').
    # TODO case 3: tool raised -> ToolMessage(status='error', content=repr(exc)).
    #   (each ToolMessage needs tool_call_id=call["id"])
    raise NotImplementedError("your code here")


ok = tool_call("a", "lookup", {"city": "Paris"})
print(dispatch(TOOLS, ok))  # expect a success ToolMessage with content '11000000'

[↑ Back to top](#top)

## 3. Problem 2 — The three failure modes, one by one

Run `dispatch` on three crafted calls and confirm each becomes a well-formed `ToolMessage` rather than a crash:

1. a `lookup` for a city **not in the table** (the tool **raises** `KeyError`),
2. a `calculator` with **non-arithmetic** input (the tool **raises** `ValueError`),
3. a call to a tool that **doesn't exist** (`wikipedia`).

Print each result; each should carry `status="error"`.

In [ ]:
bad_calls = [
    tool_call("b1", "lookup", {"city": "Atlantis"}),  # KeyError
    tool_call("b2", "calculator", {"expression": "two + 2"}),  # ValueError
    tool_call("b3", "wikipedia", {"query": "geese"}),  # unknown tool
]
# TODO: print dispatch(TOOLS, call) for each; confirm each result has status == "error".
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 — Watch the model recover (no crash)

Script a stub that first looks up a missing city (your `dispatch` turns the `KeyError` into a `status="error"` `ToolMessage`), then — having 'seen' the error — looks up a city that exists, then answers. Run `run_loop` and confirm it reaches **natural** termination instead of crashing. The loop turned a tool bug into a message the model could act on.

In [ ]:
recover_script = [
    tool_reply(tool_call("f1", "lookup", {"city": "Atlantis"})),  # raises -> status="error"
    tool_reply(tool_call("f2", "lookup", {"city": "Tokyo"})),  # recover
    text_reply("Atlantis isn't on file; Tokyo is about 37,000,000."),
]
# TODO: model = FakeModel(recover_script); run_loop(..., max_steps=10);
#       assert termination == 'natural'.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 — Why not dump the traceback? (written)

Your `dispatch` feeds the model `repr(exc)` — a short message — not the full Python traceback. Give **two** reasons that is the right call. (Hint: think about tokens, what the model can act on, and what a stack trace might leak.)

_Write your answer by editing this cell (double-click)._

1. _TODO_
2. _TODO_

[↑ Back to top](#top)

## 6. Problem 5 — Should the loop auto-retry? (written)

Your loop surfaces the error to the model and lets *it* decide whether to retry — the loop does **not** auto-retry. In a sentence or two: why is blanket auto-retry a poor default? (Hint: a `404 not found` is not a `503 service unavailable`; consider a tool that charges a card.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1005_lab_solutions.ipynb`. Done when: `dispatch` returns a `status="success"` `ToolMessage` for the good call and a `status="error"` `ToolMessage` (never a crash) for the missing city, the bad expression, and the unknown tool; and the recovery run reaches `"natural"` termination. You can state why a short error beats a traceback and why auto-retry is not a safe default.

[↑ Back to top](#top)